# Exploratory Data Analysis: WM_RL Dataset

This notebook explores the WM_RL (Working Memory - Right-Left) fMRI functional connectivity data and fluid intelligence labels.

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
%matplotlib inline

## 2. Load Data

In [ ]:
# Load FC data and labels
fc_data = np.load('../dataset/WM_RL/wm_rl_fc_combined.npy')
labels = np.load('../dataset/WM_RL/wm_rl_labels.npy')

print("FC Data Shape:", fc_data.shape)
print("Labels Shape:", labels.shape)
print("\nLabel columns: [sex, age, fluid_cognition]")
print("\nFC Data Type:", fc_data.dtype)
print("Labels Data Type:", labels.dtype)

## 3. Explore Dataset Structure

In [ ]:
# Create labels dataframe
labels_df = pd.DataFrame(labels, columns=['sex', 'age', 'fluid_cognition'])

print("Dataset Shape: {} samples x {} features (per label)".format(labels_df.shape[0], labels_df.shape[1]))
print("\nFirst few rows:")
print(labels_df.head())

print("\n\nData Types:")
print(labels_df.dtypes)

print("\n\nMissing Values:")
print(labels_df.isnull().sum())

## 4. Statistical Summary

In [ ]:
print("Descriptive Statistics for WM_RL Dataset:")
print(labels_df.describe())

print("\n\nDetailed Statistics:")
for col in labels_df.columns:
    print(f"\n{col}:")
    print(f"  Mean: {labels_df[col].mean():.4f}")
    print(f"  Median: {labels_df[col].median():.4f}")
    print(f"  Std Dev: {labels_df[col].std():.4f}")
    print(f"  Min: {labels_df[col].min():.4f}")
    print(f"  Max: {labels_df[col].max():.4f}")
    print(f"  Skewness: {stats.skew(labels_df[col]):.4f}")
    print(f"  Kurtosis: {stats.kurtosis(labels_df[col]):.4f}")

## 5. Visualize Distributions

In [ ]:
# Create distribution plots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle('WM_RL Dataset Distributions', fontsize=16, fontweight='bold')

for idx, col in enumerate(labels_df.columns):
    # Histogram
    axes[idx, 0].hist(labels_df[col], bins=30, edgecolor='black', alpha=0.7)
    axes[idx, 0].set_title(f'{col} - Histogram')
    axes[idx, 0].set_xlabel(col)
    axes[idx, 0].set_ylabel('Frequency')
    
    # Box plot
    axes[idx, 1].boxplot(labels_df[col])
    axes[idx, 1].set_title(f'{col} - Box Plot')
    axes[idx, 1].set_ylabel(col)
    
    # KDE plot
    labels_df[col].plot(kind='kde', ax=axes[idx, 2])
    axes[idx, 2].set_title(f'{col} - KDE Plot')
    axes[idx, 2].set_xlabel(col)

plt.tight_layout()
plt.show()

print("Distribution plots created successfully!")

## 6. Analyze Correlations

In [ ]:
# Compute correlation matrix
corr_matrix = labels_df.corr()

print("Correlation Matrix:")
print(corr_matrix)

# Visualize correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, square=True, 
            linewidths=1, cbar_kws={"shrink": 0.8}, fmt='.3f')
plt.title('WM_RL Dataset - Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Functional Connectivity Matrix Analysis

In [ ]:
# Analyze FC matrices
print("FC Data Analysis:")
print(f"Number of subjects: {fc_data.shape[0]}")
print(f"FC matrix shape per subject: {fc_data.shape[1:]}")
print(f"Total elements per FC matrix: {fc_data[0].size}")

# Statistics of FC values
print(f"\nFC Data Statistics:")
print(f"Mean: {fc_data.mean():.6f}")
print(f"Median: {np.median(fc_data):.6f}")
print(f"Std Dev: {fc_data.std():.6f}")
print(f"Min: {fc_data.min():.6f}")
print(f"Max: {fc_data.max():.6f}")

# Visualize a sample FC matrix
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i in range(3):
    im = axes[i].imshow(fc_data[i, 0], cmap='jet')
    axes[i].set_title(f'Subject {i} - FC Matrix')
    plt.colorbar(im, ax=axes[i])

plt.suptitle('Sample FC Matrices - WM_RL Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Identify Outliers and Anomalies

In [ ]:
# Detect outliers using IQR method
def find_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detect outliers using Z-score
def find_outliers_zscore(data, column, threshold=3):
    z_scores = np.abs(stats.zscore(data[column]))
    outliers = data[z_scores > threshold]
    return outliers

print("Outlier Detection Results:\n")
for col in labels_df.columns:
    print(f"\n{col.upper()}:")
    
    # IQR method
    outliers_iqr, lower, upper = find_outliers_iqr(labels_df, col)
    print(f"  IQR Method: {len(outliers_iqr)} outliers found")
    print(f"    Bounds: [{lower:.4f}, {upper:.4f}]")
    
    # Z-score method
    outliers_zscore = find_outliers_zscore(labels_df, col)
    print(f"  Z-score Method: {len(outliers_zscore)} outliers found (threshold=3)")

In [ ]:
# Visualize outliers
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, col in enumerate(labels_df.columns):
    outliers, lower, upper = find_outliers_iqr(labels_df, col)
    
    axes[idx].scatter(range(len(labels_df)), labels_df[col], alpha=0.6, label='Normal')
    axes[idx].scatter(outliers.index, outliers[col], color='red', label='Outliers', s=100)
    axes[idx].axhline(y=lower, color='g', linestyle='--', alpha=0.5, label='IQR Bounds')
    axes[idx].axhline(y=upper, color='g', linestyle='--', alpha=0.5)
    
    axes[idx].set_title(f'{col} - Outlier Detection')
    axes[idx].set_xlabel('Sample Index')
    axes[idx].set_ylabel(col)
    axes[idx].legend()

plt.suptitle('Outlier Detection using IQR Method - WM_RL Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Compare WM_LR and WM_RL Datasets

In [ ]:
# Load WM_LR data for comparison
wm_lr_labels = np.load('../dataset/WM_LR/wm_lr_labels.npy')
wm_lr_df = pd.DataFrame(wm_lr_labels, columns=['sex', 'age', 'fluid_cognition'])

print("Comparison of WM_LR and WM_RL Datasets:\n")
print("="*70)
print(f"{'Metric':<20} {'WM_LR':<25} {'WM_RL':<25}")
print("="*70)

for col in ['age', 'fluid_cognition']:
    print(f"\n{col.upper()}")
    print(f"  {'Mean':<18} {wm_lr_df[col].mean():<25.4f} {labels_df[col].mean():<25.4f}")
    print(f"  {'Std Dev':<18} {wm_lr_df[col].std():<25.4f} {labels_df[col].std():<25.4f}")
    print(f"  {'Min':<18} {wm_lr_df[col].min():<25.4f} {labels_df[col].min():<25.4f}")
    print(f"  {'Max':<18} {wm_lr_df[col].max():<25.4f} {labels_df[col].max():<25.4f}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age comparison
axes[0].hist(wm_lr_df['age'], bins=20, alpha=0.6, label='WM_LR', edgecolor='black')
axes[0].hist(labels_df['age'], bins=20, alpha=0.6, label='WM_RL', edgecolor='black')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Age Distribution - WM_LR vs WM_RL')
axes[0].legend()

# Fluid cognition comparison
axes[1].hist(wm_lr_df['fluid_cognition'], bins=20, alpha=0.6, label='WM_LR', edgecolor='black')
axes[1].hist(labels_df['fluid_cognition'], bins=20, alpha=0.6, label='WM_RL', edgecolor='black')
axes[1].set_xlabel('Fluid Cognition')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Fluid Cognition Distribution - WM_LR vs WM_RL')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Summary and Key Findings

In [ ]:
print("\n" + "="*60)
print("WM_RL Dataset - EDA Summary")
print("="*60)

print(f"\nDataset Size: {len(labels_df)} subjects")
print(f"FC Matrix Size: {fc_data.shape[1:]} (264x264)")

print("\nLabel Statistics:")
print(f"  Sex: {labels_df['sex'].unique()} (binary)")
print(f"  Age: {labels_df['age'].min():.1f} - {labels_df['age'].max():.1f} years (mean: {labels_df['age'].mean():.1f})")
print(f"  Fluid Cognition: {labels_df['fluid_cognition'].min():.1f} - {labels_df['fluid_cognition'].max():.1f} (mean: {labels_df['fluid_cognition'].mean():.1f})")

print("\nKey Correlations with Fluid Cognition:")
fluid_corr = labels_df.corr()['fluid_cognition'].drop('fluid_cognition').sort_values(key=abs, ascending=False)
for var, corr in fluid_corr.items():
    print(f"  {var}: {corr:.4f}")

print("\nComparison with WM_LR:")
print(f"  Number of subjects in WM_LR: {len(wm_lr_df)}")
print(f"  Number of subjects in WM_RL: {len(labels_df)}")

print("\n" + "="*60)